In [1]:
# ============================================================
# Real-Time Multi-Object Colorization using DeepLabV3 + PyQt5
# ============================================================


import sys
import cv2
import torch
import numpy as np
from PIL import Image
from datetime import datetime

from PyQt5.QtWidgets import (
    QApplication,
    QWidget,
    QLabel,
    QPushButton,
    QFileDialog,
    QVBoxLayout,
    QHBoxLayout,
    QMessageBox
)

from PyQt5.QtGui import QImage, QPixmap
from PyQt5.QtCore import QTimer

import torchvision.transforms as T
from torchvision import models



VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat',
    'bottle', 'bus', 'car', 'cat', 'chair',
    'cow', 'diningtable', 'dog', 'horse',
    'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor'
]



CLASS_COLORS = {
    'background': (0, 0, 0),
    'aeroplane': (255, 0, 0),
    'bicycle': (0, 128, 255),
    'bird': (255, 255, 0),
    'boat': (255, 128, 0),
    'bottle': (128, 255, 0),
    'bus': (0, 255, 0),
    'car': (255, 0, 255),
    'cat': (255, 255, 255),
    'chair': (128, 128, 255),
    'cow': (0, 255, 255),
    'diningtable': (255, 100, 100),
    'dog': (100, 255, 100),
    'horse': (100, 100, 255),
    'motorbike': (255, 50, 50),
    'person': (255, 255, 0),
    'pottedplant': (0, 200, 0),
    'sheep': (200, 200, 255),
    'sofa': (255, 128, 128),
    'train': (128, 0, 255),
    'tvmonitor': (50, 50, 255)
}

# ============================================================
# LOAD DEEPLABV3 MODEL
# ============================================================

print("Loading DeepLabV3 MobileNetV3 Model...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = models.segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT

model = models.segmentation.deeplabv3_mobilenet_v3_large(
    weights=weights
)

model.eval()
model.to(device)

print(f"Using device: {device}")

# ============================================================
# SEGMENTATION + COLORIZATION FUNCTION
# ============================================================



def segment_and_colorize(frame):

    transform = T.Compose([
    T.ToTensor(), 
    T.Normalize( 
        mean = [0.485, 0.456, 0.406], 
        std =  [0.229, 0.224, 0.225] )
    ])

    frame_resized = cv2.resize(frame, (768, 512))

    rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

    pil_image = Image.fromarray(rgb)

    input_tensor = transform(pil_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)['out'][0]

    predictions = output.argmax(0).cpu().numpy()

    color_mask = np.zeros_like(frame_resized)

    detected_objects = []

    for class_index, class_name in enumerate(VOC_CLASSES):

        if class_name not in CLASS_COLORS:
            continue

        mask = predictions == class_index

        if np.sum(mask) < 500:
            continue

        color_mask[mask] = CLASS_COLORS[class_name]

        detected_objects.append(class_name)

        ys, xs = np.where(mask)

        if len(xs) > 0:

            cx = int(np.mean(xs))
            cy = int(np.mean(ys))

            cv2.putText(
                color_mask,
                class_name,
                (cx, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

    blended = cv2.addWeighted(
        frame_resized,
        0.25,
        color_mask,
        0.75,
        0
    )

    return blended, detected_objects

# ============================================================
# MAIN GUI CLASS
# ============================================================

class VideoColorizationApp(QWidget):

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Real-Time Multi-Object Colorization")
        self.setGeometry(100, 50, 1300, 850)

        # ---------------- DARK THEME ----------------
        self.setStyleSheet("""
        QWidget{
           background-color:#1E1E1E;
           color:white;
           font-size:13px;
        }

        QPushButton{
           background-color:#3498db;
           color:white;
           border-radius:15px;
           padding:10px;
           font-weight:bold;
        }

        QPushButton:hover{
           background-color:#2980b9;
        }

        QLabel{
           color:white;
        }
        """)

        
        self.title_label = QLabel(
          "Real-Time Multi-Object Colorization"
        )

        self.title_label.setStyleSheet("""
          font-size:24px;
          font-weight:bold;
          color:#00FFFF;
         """)

     
        self.video_label = QLabel()

        self.video_label.setFixedSize(1000, 600)

        self.video_label.setStyleSheet("""
            border:3px solid cyan;
            background:black;
        """)

   
        self.webcam_btn = QPushButton("Start Webcam")
        self.video_btn = QPushButton("Upload Video")
        self.stop_btn = QPushButton("Stop")
        self.save_btn = QPushButton("Save Frame")

    
        self.fps_label = QLabel("FPS : 0")

        self.device_label = QLabel(
            f"Device : {device}"
        )

        self.object_label = QLabel(
            "Objects : None"
        )

        self.frame_label = QLabel(
            "Frame : 0"
        )

        self.status_label = QLabel(
            "Ready"
        ) 

        self.status_label.setStyleSheet("""
            color:yellow;
            font-size:15px;
        """)

        
        button_layout = QHBoxLayout()

        button_layout.addWidget(self.webcam_btn)
        button_layout.addWidget(self.video_btn)
        button_layout.addWidget(self.stop_btn)
        button_layout.addWidget(self.save_btn)

       
        info_layout = QHBoxLayout()

        info_layout.addWidget(self.fps_label)
        info_layout.addWidget(self.device_label)
        info_layout.addWidget(self.object_label)
        info_layout.addWidget(self.frame_label)

        
        main_layout = QVBoxLayout()

        main_layout.addWidget(
            self.title_label
        )

        main_layout.addWidget(
            self.video_label
        )

        main_layout.addLayout(
            button_layout
        )

        main_layout.addLayout(
            info_layout
        )

        main_layout.addWidget(
            self.status_label
        )

        self.setLayout(main_layout)

        # ---------------- TIMER ----------------
        self.timer = QTimer()

        self.timer.timeout.connect(
            self.update_frame
        )

        self.cap = None

        self.prev_time = 0

        self.frame_count = 0

        
        self.webcam_btn.clicked.connect(
            self.start_webcam
        )

        self.video_btn.clicked.connect(
            self.load_video
        )

        self.stop_btn.clicked.connect(
            self.stop_video
        )

        self.save_btn.clicked.connect(
            self.save_frame
        )

    def start_webcam(self):

        self.cap = cv2.VideoCapture(0)

        if not self.cap.isOpened():
            QMessageBox.critical(self, "Error", "Cannot access webcam")
            return

        self.timer.start(30)

    # ========================================================
    # LOAD VIDEO
    # ========================================================

    def load_video(self):

        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open Video",
            "",
            "Video Files (*.mp4 *.avi *.mov)"
        )

        if file_path:

            self.cap = cv2.VideoCapture(file_path)

            self.timer.start(30)

    # ========================================================
    # STOP VIDEO
    # ========================================================

    def stop_video(self):

        self.timer.stop()

        if self.cap:
            self.cap.release()

    def save_frame(self):

        if hasattr(self, "current_frame"):

            cv2.imwrite(
                "saved_frame.jpg",
                self.current_frame
            )

            QMessageBox.information(
                self,
                "Saved",
                "Frame saved successfully!"
            )

    # ========================================================
    # UPDATE FRAME
    # ========================================================

    def update_frame(self):

        ret, frame = self.cap.read()

        if not ret:
            self.stop_video()
            return

        # Process frame
        output_frame, detected_objects = segment_and_colorize(frame)

        self.current_frame = output_frame.copy()

        self.frame_count += 1

        self.frame_label.setText(
            f"Frame : {self.frame_count}"
        )

        self.object_label.setText(
            "Objects : " +
            ", ".join(detected_objects[:5])
        )

        # FPS Calculation
        current_time = cv2.getTickCount()

        fps = cv2.getTickFrequency() / (
            current_time - self.prev_time
        ) if self.prev_time != 0 else 0

        self.prev_time = current_time

        self.fps_label.setText(f"FPS: {fps:.2f}")

        current_time = datetime.now().strftime(
            "%H:%M:%S"
        )

        cv2.putText(
            output_frame,
            current_time,
            (850,30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,255,255),
            2
        )

        if detected_objects:

          cv2.putText(
              output_frame,
              "Detected: " + ", ".join(detected_objects[:5]),
              (10, 30),
              cv2.FONT_HERSHEY_SIMPLEX,
              0.7,
              (255, 255, 255),
              2
          )

        
        rgb_image = cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB)

        h, w, ch = rgb_image.shape

        bytes_per_line = ch * w

        qt_image = QImage(
            rgb_image.data,
            w,
            h,
            bytes_per_line,
            QImage.Format_RGB888
        )

        pixmap = QPixmap.fromImage(qt_image)

        self.video_label.setPixmap(pixmap)

# ============================================================
# MAIN APPLICATION
# ============================================================

if __name__ == "__main__":

    app = QApplication(sys.argv)

    window = VideoColorizationApp()

    window.show()

    sys.exit(app.exec_())

Loading DeepLabV3 MobileNetV3 Model...
Using device: cpu


SystemExit: 0

C:\Users\SERVER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
